# 01 · Data, model and evaluation

Telecom customer churn (IBM Telco Customer Churn). This notebook covers the problem definition, the data,
the evaluation protocol and the models we will explain later. The logic lives in `src/`; the notebook only
calls it and comments on the results.

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd
from sklearn.base import clone

sys.path.insert(0, str(Path.cwd().parent))  # make `src` importable when the notebook runs from notebooks/
from src.data import download, load_clean
from src.modeling import MAX_AUC_GAP, SEED, SUSPICIOUS_AUC, cv_and_gap, make_models, split

## 1. Task and data

**Task.** Binary classification. Input: the profile of a customer (contract, services, billing, tenure).
Output: whether the customer churned (1) or not (0). **Stakeholder.** The retention team, which has a limited
budget and decides whom to contact and with which offer.

**Data.** IBM sample data set, 7,043 customers x 21 columns, downloaded by `python -m src.data`.

In [2]:
X, y = load_clean(download())
print(f"{len(X)} customers, {X.shape[1]} features, churn rate {y.mean():.4f}")
print(f"numeric: {list(X.select_dtypes('number').columns)}")
print(f"categorical: {X.select_dtypes(exclude='number').shape[1]} columns of text")

7043 customers, 19 features, churn rate 0.2654
numeric: ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']
categorical: 15 columns of text


In [3]:
# Churn is driven by the type of contract long before any model is fitted.
(pd.concat([X["Contract"], y], axis=1).groupby("Contract")["Churn"]
   .agg(customers="size", churn_rate="mean").round(3))

,customers,churn_rate
Contract,,
Month-to-month,3875,0.427
One year,1473,0.113
Two year,1695,0.028


**Key preprocessing.**
- `customerID` is dropped: it identifies a row, it is not a feature.
- `TotalCharges` is text in the raw file because 11 rows are blank. All 11 are customers with `tenure = 0`
  (signed up, not billed yet), so 0 is their true value; `load_clean` refuses any other blank.
- This cleaning is row-wise and fits nothing, so it cannot leak. Everything that *is* fitted (one-hot encoding,
  scaling for the linear model) lives inside the sklearn `Pipeline`, and is refitted on every training fold.

In [4]:
duplicates = pd.concat([X, y], axis=1).duplicated().sum()
print(f"{duplicates} rows are exact duplicates of another row (customerID aside)")

22 rows are exact duplicates of another row (customerID aside)


The duplicates are kept: they are plausible different customers with identical profiles, and dropping them
would change the population. A duplicate may fall on both sides of the split, which makes the test estimate
very slightly optimistic (22 of 7,043 rows).

## 2. Evaluation protocol

- **Split.** Stratified 80/20 with a fixed seed. Telco is cross-sectional (one row per customer, no time axis),
  so a random split is appropriate; a time-aware split would need dates we do not have.
- **Selection.** Hyperparameters are chosen by 5-fold CV **on the training set only**. The CV number reported is
  *nested*: the search is redone inside every outer fold, so it is not inflated by picking the best
  configuration on the same folds.
- **Test set.** Only scored, never used to fit or tune.
- **Imbalance.** 26.5 % churn. No resampling and no class weights: the decision threshold and the
  churn-class metrics deal with it (next section), which avoids distorting the probabilities.
- **"Not overfitted".** Defined before tuning as train AUC minus CV AUC below `MAX_AUC_GAP`.

In [5]:
X_train, X_test, y_train, y_test = split(X, y)
print(f"train {len(X_train)} (churn {y_train.mean():.4f}) | test {len(X_test)} (churn {y_test.mean():.4f})")
print(f"max allowed train-CV AUC gap: {MAX_AUC_GAP}")

train 5634 (churn 0.2654) | test 1409 (churn 0.2654)
max allowed train-CV AUC gap: 0.03


## 3. Models

1. **Majority class.** Always predicts the prior: the floor every model must beat.
2. **Logistic regression** (L2, tuned `C`, standardised numerics). The interpretable reference: its coefficients
   are an explanation we can compare against.
3. **XGBoost** (tuned depth and number of trees, `subsample` and `colsample_bytree` 0.8). The main model, whose
   explanations we will study with TreeSHAP.

In [6]:
models = make_models()
table = pd.DataFrame({name: cv_and_gap(est, X_train, y_train, X_test, y_test)
                      for name, est in models.items()}).T
table.round(4)

,train_auc,cv_auc,cv_std,test_auc,gap,cv_pr_auc,test_pr_auc
majority,0.500,0.5000,0.0000,0.5000,0.0000,0.2654,0.2654
logistic,0.850,0.8463,0.0129,0.8413,0.0037,0.6607,0.6285
xgboost,0.859,0.8508,0.0117,0.8481,0.0082,0.6720,0.6699


In [7]:
# Guardrails. A failure here stops the notebook instead of hiding a problem.
tuned = table.loc[["logistic", "xgboost"]]
assert (tuned["gap"] < MAX_AUC_GAP).all(), f"overfitting: gap above {MAX_AUC_GAP}"
assert (table["cv_auc"] < SUSPICIOUS_AUC).all(), f"CV AUC above {SUSPICIOUS_AUC}: rule out leakage first"
assert (tuned["cv_auc"] > table.loc["majority", "cv_auc"] + 0.2).all(), "models barely beat the majority baseline"
print("checks passed: no overfitting gap, no suspicious AUC, both models clearly above the majority baseline")

checks passed: no overfitting gap, no suspicious AUC, both models clearly above the majority baseline


In [8]:
# Which configuration wins when the search sees the whole training set.
final = {name: clone(est).fit(X_train, y_train) for name, est in models.items() if name != "majority"}
{name: model.best_params_ for name, model in final.items()}

{'logistic': {'model__C': 10},
 'xgboost': {'model__max_depth': 1, 'model__n_estimators': 600}}

### Reading the results

- **In the expected range, no sign of leakage.** CV AUC is 0.846 (logistic) and 0.851 (XGBoost). Published
  work on this data set reports about 0.84-0.86; an AUC above 0.90 would have been treated as leakage.
- **Not overfitted.** The train-CV gap is 0.004 and 0.008, between 4 and 8 times below the 0.03 we allowed,
  and the test AUC (0.841, 0.848) is within one fold standard deviation (0.012) of the CV estimate.
- **The churn class is being found.** PR-AUC of a model with no skill equals the churn rate, 0.265 (the
  majority row). Both models reach 0.66-0.67 in CV.
- **XGBoost is the main model, but the margin is small.** Its ROC-AUC is ahead by 0.005 in CV, less than the
  spread between folds (0.012), so on ROC-AUC alone the two models are not separable. It is ahead on PR-AUC
  too (CV 0.672 vs 0.661, test 0.670 vs 0.629), but we have not measured the fold spread of PR-AUC, so we do
  not claim more than "never worse". Logistic regression stays as the interpretable reference.
- **The best XGBoost is a set of depth-1 trees (stumps).** Boosted stumps are an *additive* model: each feature
  contributes on its own, with no interactions. So on this data set interactions are not needed to predict
  well. *Inference to check in the XAI notebook:* SHAP values of such a model are the per-feature effects
  themselves, so LR and XGBoost should tell a similar story.
- **Grid edges.** The winners sit at the edge of the grid (depth 1 is the lowest possible value; 600 trees the
  highest tried; `C` 10-100). We widened the grid twice using training folds only (and switched to one column
  per Yes/No feature): CV AUC went 0.8503 -> 0.8505 -> 0.8508. The AUC surface is a plateau, with differences
  far below the fold-to-fold noise, so we stopped tuning.
- **AUC is only the first look.** With 26.5 % churn the decision depends on the threshold, and accuracy alone
  is misleading (a model that never predicts churn is 73.5 % accurate). Churn-class metrics come next.

In [9]:
out = Path.cwd().parent / "reports" / "metrics.json"
out.parent.mkdir(exist_ok=True)
out.write_text(json.dumps({"seed": SEED, "auc": table.round(4).to_dict("index")}, indent=2))
print("saved", out.name)

saved metrics.json
